In [ ]:
"""
WRDS data pull for S&P 500 spinoff index arbitrage project.

Pulls three things:
  1. S&P 500 point-in-time constituent history (Compustat IDXCST_HIS)
  2. CRSP <-> Compustat link table (CCM)
  3. CRSP daily prices, returns, and volume for a given permno list

Run order: pull_sp500_constituents() -> pull_ccm_link() -> pull_crsp_daily()
The constituent + link tables tell you WHICH securities to pull prices for,
so don't skip straight to step 3.

Requires a WRDS account with access to crsp and comp libraries.
First run will prompt for WRDS username/password and offer to save
credentials to ~/.pgpass.
"""

import wrds
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


def get_connection():
    """Open a WRDS connection. Prompts for credentials on first use."""
    return wrds.Connection()


def pull_sp500_constituents(db, start_date="1996-01-01"):
    """
    Point-in-time S&P 500 membership from Compustat.

    'from' = date added to the index
    'thru' = date removed (NULL if still a current member)

    gvkeyx = '000003' is the Compustat index code for the S&P 500.

    Note: idxcst_his only carries the membership keys, not a company
    name column. If you want names attached, join to comp.names (or
    comp.company) on gvkey separately — see attach_company_names().
    """
    df = db.raw_sql(f"""
        SELECT
            gvkey,
            "from"  AS start_date,
            thru    AS end_date
        FROM comp.idxcst_his
        WHERE gvkeyx = '000003'
          AND "from" >= '{start_date}'
        ORDER BY "from"
    """, date_cols=["start_date", "end_date"])

    # still-active members have a NULL end_date — fill with today so
    # downstream date-range filtering doesn't choke on NaT
    df["end_date"] = df["end_date"].fillna(pd.Timestamp.today().normalize())
    df["still_active"] = df["end_date"] == pd.Timestamp.today().normalize()

    df.to_parquet(RAW_DIR / "sp500_constituents_pit.parquet")
    print(f"[sp500_constituents] {len(df)} membership records, "
          f"{df['gvkey'].nunique()} unique companies")
    return df


def inspect_table_columns(db, schema, table):
    """
    Quick schema lookup — useful whenever a WRDS table throws an
    UndefinedColumn error and you need to see what's actually there
    before guessing again.

    Example: inspect_table_columns(db, 'comp', 'idxcst_his')
    """
    df = db.raw_sql(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name = '{table}'
        ORDER BY ordinal_position
    """)
    print(df.to_string(index=False))
    return df


def attach_company_names(db, constituents: pd.DataFrame) -> pd.DataFrame:
    """
    Join company identifiers onto the constituent history via comp.names.

    Confirmed schema (your WRDS instance): comp.names is a flat,
    one-row-per-gvkey table — no date-ranged history. Columns:
    gvkey, conm, tic, cusip, cik, sic, naics, gsubind, gind,
    year1, year2 (year1/year2 are the fiscal years the name was
    current, not exact dates — not used for the as-of join here
    since there's nothing to join against at daily granularity).

    Because it's one row per gvkey, this is a flat merge, not an
    as-of join. If a company changed its name over its history,
    you'll only get whichever name WRDS has on file as current/latest
    — fine for our purposes since we mainly need cusip/tic for
    cross-referencing against Bloomberg, not the historical name itself.
    """
    gvkeys = constituents["gvkey"].unique().tolist()
    gvkey_str = ",".join(f"'{g}'" for g in gvkeys)

    names = db.raw_sql(f"""
        SELECT gvkey, conm AS company_name, tic AS ticker, cusip, cik, sic, naics
        FROM comp.names
        WHERE gvkey IN ({gvkey_str})
    """)
    names = names.drop_duplicates(subset=["gvkey"], keep="last")

    return constituents.merge(names, on="gvkey", how="left")


def pull_ccm_link(db):
    """
    CRSP-Compustat merged link table.

    Compustat identifies firms by gvkey, CRSP identifies securities by
    permno. You need this table to go from one to the other. Filtering
    to linktype LU/LC and linkprim P/C keeps only the primary link per
    company (avoids double-counting dual-class shares etc).
    """
    df = db.raw_sql("""
        SELECT
            gvkey,
            lpermno  AS permno,
            linktype,
            linkprim,
            linkdt,
            linkenddt
        FROM crsp.ccmxpf_lnkhist
        WHERE linktype IN ('LU', 'LC')
          AND linkprim IN ('P', 'C')
    """, date_cols=["linkdt", "linkenddt"])

    df["linkenddt"] = df["linkenddt"].fillna(pd.Timestamp.today().normalize())

    df.to_parquet(RAW_DIR / "ccm_link.parquet")
    print(f"[ccm_link] {len(df)} link records, {df['permno'].nunique()} unique permnos")
    return df


def map_gvkey_to_permno(constituents: pd.DataFrame, link: pd.DataFrame) -> pd.DataFrame:
    """
    Join constituent history to the link table on gvkey, keeping only
    links that were active during the membership window. A gvkey can
    have more than one permno over time (rare, but happens around
    restructurings), so this is an as-of join, not a flat merge.
    """
    merged = constituents.merge(link, on="gvkey", how="left")

    # keep only rows where the CCM link was valid at some point during
    # the constituent's index membership window
    overlap = (merged["linkdt"] <= merged["end_date"]) & (
        merged["linkenddt"] >= merged["start_date"]
    )
    merged = merged[overlap].copy()

    merged = merged.drop_duplicates(subset=["gvkey", "start_date", "permno"])
    merged.to_parquet(RAW_DIR / "sp500_constituents_with_permno.parquet")
    print(f"[gvkey_to_permno] {len(merged)} rows after mapping, "
          f"{merged['permno'].nunique()} unique permnos")
    return merged


def pull_crsp_daily(db, permnos, start_date="1996-01-01", end_date=None,
                     chunk_size=500):
    """
    Daily prices, returns, volume, and shares outstanding from CRSP DSF.

    Pulls in chunks because passing thousands of permnos in a single
    IN clause is slow and sometimes hits query size limits.

    prc is negative when it's a bid-ask midpoint rather than an actual
    trade price (happens for illiquid names) — take abs() before using.
    cfacpr/cfacshr are cumulative adjustment factors for splits; divide
    prc by cfacpr to get a split-adjusted price series.
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    permnos = sorted(set(int(p) for p in permnos if pd.notna(p)))
    chunks = [permnos[i:i + chunk_size] for i in range(0, len(permnos), chunk_size)]

    frames = []
    for i, chunk in enumerate(chunks):
        permno_str = ",".join(str(p) for p in chunk)
        df = db.raw_sql(f"""
            SELECT
                permno,
                date,
                prc,
                ret,
                retx,
                vol,
                shrout,
                cfacpr,
                cfacshr
            FROM crsp.dsf
            WHERE permno IN ({permno_str})
              AND date BETWEEN '{start_date}' AND '{end_date}'
            ORDER BY permno, date
        """, date_cols=["date"])
        frames.append(df)
        print(f"[crsp_daily] chunk {i + 1}/{len(chunks)} -> {len(df)} rows")

    daily = pd.concat(frames, ignore_index=True)

    # derived fields used throughout the project
    daily["adj_prc"] = daily["prc"].abs() / daily["cfacpr"]
    daily["mktcap"] = daily["prc"].abs() * daily["shrout"] * 1_000  # shrout is in thousands
    daily["dollar_vol"] = daily["vol"] * daily["prc"].abs()

    daily.to_parquet(RAW_DIR / "crsp_daily.parquet")
    print(f"[crsp_daily] total {len(daily)} rows, {daily['permno'].nunique()} permnos")
    return daily


def pull_crsp_index_returns(db, start_date="1996-01-01"):
    """S&P 500 / market index daily returns, needed for beta estimation."""
    df = db.raw_sql(f"""
        SELECT date, vwretd, ewretd, sprtrn
        FROM crsp.dsi
        WHERE date >= '{start_date}'
        ORDER BY date
    """, date_cols=["date"])

    df.to_parquet(RAW_DIR / "crsp_index_returns.parquet")
    print(f"[index_returns] {len(df)} rows")
    return df


def compute_adv(daily: pd.DataFrame, window=30) -> pd.DataFrame:
    """
    Rolling average daily dollar volume per permno — one of the core
    features for the deletion probability model. Computed here rather
    than in the modeling notebook so it lives next to the rest of the
    raw-to-interim cleaning logic.
    """
    daily = daily.sort_values(["permno", "date"]).copy()
    daily["adv_30d"] = (
        daily.groupby("permno")["dollar_vol"]
        .transform(lambda x: x.rolling(window, min_periods=5).mean())
    )
    return daily


if __name__ == "__main__":
    db = get_connection()

    constituents = pull_sp500_constituents(db)
    constituents = attach_company_names(db, constituents)  # optional, comment out if not needed
    constituents.to_parquet(RAW_DIR / "sp500_constituents_pit.parquet")

    link = pull_ccm_link(db)
    mapped = map_gvkey_to_permno(constituents, link)

    permnos = mapped["permno"].dropna().unique()
    daily = pull_crsp_daily(db, permnos)
    daily = compute_adv(daily)
    daily.to_parquet(RAW_DIR / "crsp_daily.parquet")  # overwrite with adv_30d added

    pull_crsp_index_returns(db)

    db.close()
    print("Done. Raw files written to data/raw/")

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
[sp500_constituents] 357 membership records, 354 unique companies


/var/folders/g1/_6qfg71x03x35bdydk63jnfm0000gp/T/ipykernel_8253/1126608829.py:57: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["end_date"] = df["end_date"].fillna(pd.Timestamp.today().normalize())
/var/folders/g1/_6qfg71x03x35bdydk63jnfm

[ccm_link] 33324 link records, 29567 unique permnos
[gvkey_to_permno] 360 rows after mapping, 358 unique permnos
[crsp_daily] chunk 1/1 -> 1978546 rows


/var/folders/g1/_6qfg71x03x35bdydk63jnfm0000gp/T/ipykernel_8253/1126608829.py:211: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  daily["adj_prc"] = daily["prc"].abs() / daily["cfacpr"]
/var/folders/g1/_6qfg71x03x35bdydk63jnfm0000gp/T/ipykern

[crsp_daily] total 1978546 rows, 356 permnos


/var/folders/g1/_6qfg71x03x35bdydk63jnfm0000gp/T/ipykernel_8253/1126608829.py:242: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  daily["adv_30d"] = (


[index_returns] 7300 rows
Done. Raw files written to data/raw/


In [4]:
inspect_table_columns(db, 'comp', 'names')

column_name         data_type
      gvkey character varying
       conm character varying
        tic character varying
      cusip character varying
        cik character varying
        sic character varying
      naics character varying
    gsubind character varying
       gind character varying
      year1  double precision
      year2  double precision


,column_name,data_type
0,gvkey,character varying
1,conm,character varying
2,tic,character varying
3,cusip,character varying
4,cik,character varying
5,sic,character varying
6,naics,character varying
7,gsubind,character varying
8,gind,character varying
9,year1,double precision


In [ ]:
"""Load raw parquet files written by pull_data.ipynb."""

from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")

RAW_FILES = {
    "constituents": RAW_DIR / "sp500_constituents_pit.parquet",
    "constituents_with_permno": RAW_DIR / "sp500_constituents_with_permno.parquet",
    "ccm_link": RAW_DIR / "ccm_link.parquet",
    "crsp_daily": RAW_DIR / "crsp_daily.parquet",
    "index_returns": RAW_DIR / "crsp_index_returns.parquet",
}


def load_raw_data() -> dict[str, pd.DataFrame]:
    """Read all raw parquet files into a dict of DataFrames."""
    missing = [name for name, path in RAW_FILES.items() if not path.exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing raw files: {missing}. Run the WRDS pull cell first."
        )

    return {name: pd.read_parquet(path) for name, path in RAW_FILES.items()}


def summarize_raw_data(data: dict[str, pd.DataFrame]) -> None:
    """Print shape and date range for each loaded table."""
    for name, df in data.items():
        line = f"[{name}] {len(df):,} rows x {len(df.columns)} cols"
        for col in ("date", "start_date"):
            if col in df.columns:
                line += f" | {col}: {df[col].min().date()} -> {df[col].max().date()}"
        print(line)


data = load_raw_data()
summarize_raw_data(data)

constituents = data["constituents"]
constituents_with_permno = data["constituents_with_permno"]
ccm_link = data["ccm_link"]
crsp_daily = data["crsp_daily"]
index_returns = data["index_returns"]

constituents.head()